# Citi Bike Demand Project — Full Data Pipeline

This notebook implements the complete data workflow:

| Step | Description |
|------|-------------|
| 0 | Configuration & imports |
| 1 | Web parsing — find monthly zip links from the public Citi Bike index |
| 2 | Download — save Jan 2026 & Feb 2026 raw zip files to disk |
| 3 | Weather — fetch NYC daily weather from Open-Meteo (no API key needed) |
| 4 | Load & Clean — one month at a time to keep memory low |
| 5 | Aggregate — build a station-day panel per month, then combine |
| 6 | Filter — keep only the top 300 busiest start stations |
| 7 | Merge — attach daily weather to every station-day row |
| 8 | Text preprocessing — keyword indicators from station name |
| 9 | Save — write `citibike_station_day_clean.csv` |

**Final output:** `output/citibike_station_day_clean.csv`  
**Approximate size:** 300 stations × 59 days ≈ 17,700 rows

---
### Required packages
Run the cell below once if you have not installed these yet.

In [1]:
# Run this cell once to install dependencies
# You can skip it if the packages are already installed
%pip install requests beautifulsoup4 lxml pandas numpy

---
## Step 0 — Imports & Configuration

All tunable parameters are collected here so you only need to edit one cell if anything changes:
- `MONTHS` — which monthly files to download
- `TOP_N_STATIONS` — how many busiest stations to keep
- `WEATHER_START / WEATHER_END` — date range for weather data
- `RAW_DIR / OUT_DIR` — where files are saved

In [2]:
import os
import re
import zipfile
from pathlib import Path

# Third-party
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np

# Configuration
RAW_DIR  = Path("raw_data")   # downloaded zip files saved here
OUT_DIR  = Path("output")     # final CSV saved here
RAW_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

MONTHS    = ["202601", "202602"]          # January and February 2026
INDEX_URL = "https://s3.amazonaws.com/tripdata/index.html"

# Only load these columns from each trip CSV (keeps memory footprint small)
NEEDED_COLS = [
    "ride_id",
    "started_at",
    "ended_at",
    "start_station_id",
    "start_station_name",
    "start_lat",
    "start_lng",
    "member_casual",
    "rideable_type",
]

TOP_N_STATIONS = 300        # keep the 300 busiest start stations

# NYC coordinates and date range for the Open-Meteo weather API
NYC_LAT       = 40.7128
NYC_LON       = -74.0060
WEATHER_START = "2026-01-01"
WEATHER_END   = "2026-02-28"

print("Configuration loaded. Directories ready:")
print(f"  Raw data  → {RAW_DIR.resolve()}")
print(f"  Output    → {OUT_DIR.resolve()}")

Configuration loaded. Directories ready:
  Raw data  → /content/raw_data
  Output    → /content/output


---
## Step 1 — Web Parsing: Find Monthly Zip Links

The Citi Bike trip data is hosted on an S3 bucket at `https://s3.amazonaws.com/tripdata/`.  
We parse its index page with **BeautifulSoup** to extract the exact download URLs for the two monthly files.

Three strategies are tried in order:
1. **HTML `<a>` tags** — in case the index renders as a normal HTML page
2. **XML `<Key>` tags** — in case the S3 bucket returns a raw XML listing
3. **Direct URL probe** — construct the two known URL patterns and send a HEAD request to verify they exist

In [3]:
print("Parsing Citi Bike trip-data index for file links …")

response = requests.get(INDEX_URL, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "lxml")   # lxml handles XML and HTML alike

zip_links: dict[str, str] = {}

# Strategy A — HTML <a href="..."> anchors
for tag in soup.find_all("a", href=True):
    href = tag["href"]
    for month in MONTHS:
        if month in href and href.endswith(".zip"):
            url = href if href.startswith("http") \
                       else f"https://s3.amazonaws.com/tripdata/{href}"
            zip_links[month] = url

# Strategy B — raw S3 XML <Key> tags
if len(zip_links) < len(MONTHS):
    for key_tag in soup.find_all("key"):
        fname = key_tag.get_text(strip=True)
        for month in MONTHS:
            if month not in zip_links and month in fname and fname.endswith(".zip"):
                zip_links[month] = f"https://s3.amazonaws.com/tripdata/{fname}"

print(f"Links found via index: {zip_links}")

# Strategy C — fallback: probe known URL patterns directly
for month in MONTHS:
    if month in zip_links:
        continue
    candidates = [
        f"https://s3.amazonaws.com/tripdata/{month}-citibike-tripdata.csv.zip",
        f"https://s3.amazonaws.com/tripdata/{month}-citibike-tripdata.zip",
    ]
    for url in candidates:
        try:
            r = requests.head(url, timeout=10)
            if r.status_code == 200:
                zip_links[month] = url
                print(f"Fallback URL resolved for {month}: {url}")
                break
        except requests.RequestException:
            continue

if not zip_links:
    raise RuntimeError(
        "Could not resolve any download URLs. "
        "Check that the Citi Bike S3 bucket is accessible."
    )

print(f"\nFinal resolved URLs:")
for month, url in zip_links.items():
    print(f"  {month}: {url}")

Parsing Citi Bike trip-data index for file links …
Links found via index: {}
Fallback URL resolved for 202601: https://s3.amazonaws.com/tripdata/202601-citibike-tripdata.zip
Fallback URL resolved for 202602: https://s3.amazonaws.com/tripdata/202602-citibike-tripdata.zip

Final resolved URLs:
  202601: https://s3.amazonaws.com/tripdata/202601-citibike-tripdata.zip
  202602: https://s3.amazonaws.com/tripdata/202602-citibike-tripdata.zip


---
## Step 2 — Download Raw Zip Files

Each monthly zip file is streamed to disk in 1 MB chunks to avoid loading the entire file into memory at once.  
Files are saved to `raw_data/` and **skipped if already present**, so re-running this notebook is safe.

In [4]:
downloaded_zips: dict[str, Path] = {}

for month, url in zip_links.items():
    local_path = RAW_DIR / f"{month}-citibike-tripdata.zip"

    if local_path.exists():
        print(f"{month}: already on disk — skipping download.")
    else:
        print(f"{month}: downloading from {url} …")
        r = requests.get(url, stream=True, timeout=300)
        r.raise_for_status()
        with open(local_path, "wb") as fout:
            for chunk in r.iter_content(chunk_size=1024 * 1024):  # 1 MB chunks
                fout.write(chunk)
        print(f"{month}: saved → {local_path}  ({local_path.stat().st_size / 1e6:.1f} MB)")

    downloaded_zips[month] = local_path

print("\nAll files ready:", list(downloaded_zips.values()))

202601: downloading from https://s3.amazonaws.com/tripdata/202601-citibike-tripdata.zip …
202601: saved → raw_data/202601-citibike-tripdata.zip  (353.8 MB)
202602: downloading from https://s3.amazonaws.com/tripdata/202602-citibike-tripdata.zip …
202602: saved → raw_data/202602-citibike-tripdata.zip  (237.6 MB)

All files ready: [PosixPath('raw_data/202601-citibike-tripdata.zip'), PosixPath('raw_data/202602-citibike-tripdata.zip')]


---
## Step 3 — Fetch NYC Daily Weather

Weather data comes from the **Open-Meteo Historical Weather API** — it is free and requires no API key.

We request three daily variables for NYC (lat 40.71, lon −74.01) over Jan–Feb 2026:
- `temperature_2m_mean` → **mean_temperature** (°C)
- `precipitation_sum` → **precipitation_sum** (mm)
- `windspeed_10m_max` → **max_wind_speed** (km/h)

After fetching, we check for missing values and forward-fill any gaps.

In [5]:
WEATHER_URL = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={NYC_LAT}&longitude={NYC_LON}"
    f"&start_date={WEATHER_START}&end_date={WEATHER_END}"
    "&daily=temperature_2m_mean,precipitation_sum,windspeed_10m_max"
    "&timezone=America%2FNew_York"
)

wr = requests.get(WEATHER_URL, timeout=30)
wr.raise_for_status()
wj = wr.json()

weather_df = pd.DataFrame({
    "date":              pd.to_datetime(wj["daily"]["time"]),
    "mean_temperature":  wj["daily"]["temperature_2m_mean"],
    "precipitation_sum": wj["daily"]["precipitation_sum"],
    "max_wind_speed":    wj["daily"]["windspeed_10m_max"],
})

# Check for missing values and forward-fill any gaps
print("Missing value counts before fill:")
print(weather_df.isnull().sum().to_string())
weather_df.ffill(inplace=True)

print(f"\nWeather rows fetched: {len(weather_df)}")
print(f"Date range: {weather_df['date'].min().date()} → {weather_df['date'].max().date()}")
print("\nPreview:")
weather_df.head()

Missing value counts before fill:
date                 0
mean_temperature     0
precipitation_sum    0
max_wind_speed       0

Weather rows fetched: 59
Date range: 2026-01-01 → 2026-02-28

Preview:


,date,mean_temperature,precipitation_sum,max_wind_speed
0,2026-01-01,-3.4,1.1,25.1
1,2026-01-02,-4.4,0.3,14.4
2,2026-01-03,-3.5,0.0,10.3
3,2026-01-04,-2.2,0.3,12.1
4,2026-01-05,-1.8,0.8,12.6


---
## Step 4 — Cleaning Function

This function is applied to each month's raw trip DataFrame. The cleaning steps are:

| Sub-step | Operation |
|----------|-----------|
| 4a | Parse `started_at` and `ended_at` as datetimes; extract `date` |
| 4b | Drop rows missing any critical field (station ID/name, lat/lng, timestamps) |
| 4c | Remove duplicate `ride_id` values (keep first occurrence) |
| 4d | Compute `trip_duration_min`; drop trips with zero or negative duration |
| 4e | Standardize `start_station_name`: lowercase + collapse extra whitespace |

The function returns a cleaned trip-level DataFrame ready for aggregation.

In [6]:
def clean_month(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply all cleaning steps to a single month's raw trip DataFrame.
    Returns a cleaned trip-level DataFrame.
    """
    # 4a. parse timestamps, invalid strings silently become NaT
    df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
    df["ended_at"]   = pd.to_datetime(df["ended_at"],   errors="coerce")
    df["date"]       = df["started_at"].dt.normalize()   # midnight-truncated date

    # 4b. drop rows with any missing critical field
    critical = [
        "ride_id", "started_at", "ended_at",
        "start_station_id", "start_station_name",
        "start_lat", "start_lng",
    ]
    n_before = len(df)
    df.dropna(subset=critical, inplace=True)
    print(f"    Rows dropped (missing critical fields) : {n_before - len(df):,}")

    # 4c. remove duplicate ride IDs (keep first occurrence)
    n_before = len(df)
    df.drop_duplicates(subset="ride_id", keep="first", inplace=True)
    print(f"    Rows dropped (duplicate ride_id)       : {n_before - len(df):,}")

    # 4d. compute trip duration in minutes; remove non-positive durations
    df["trip_duration_min"] = (
        (df["ended_at"] - df["started_at"]).dt.total_seconds() / 60.0
    )
    n_before = len(df)
    df = df[df["trip_duration_min"] > 0].copy()
    print(f"    Rows dropped (invalid duration <= 0)   : {n_before - len(df):,}")

    # 4e. standardize station name: lowercase + collapse internal whitespace
    df["start_station_name"] = (
        df["start_station_name"]
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    return df

print("clean_month() function defined.")

clean_month() function defined.


---
## Step 5 — Aggregation Function

This function collapses cleaned trip-level rows into a **station-day panel** — one row per `(station, date)` pair.

For each group it computes:
- `daily_trip_count` — total trips starting from that station on that day
- `casual_share` / `member_share` — fraction of casual vs. member riders
- `classic_bike_share` / `electric_bike_share` — fraction by bike type
- `start_lat` / `start_lng` — **median** coordinates (smooths GPS jitter across rows)

In [7]:
def aggregate_month(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate cleaned trip-level data into a station-day panel.
    Returns one row per (start_station_id, start_station_name, date).
    """
    agg = (
        df.groupby(
            ["start_station_id", "start_station_name", "date"],
            as_index=False,
            sort=False,
        )
        .agg(
            daily_trip_count  = ("ride_id",       "count"),
            casual_count      = ("member_casual",  lambda x: (x == "casual").sum()),
            member_count      = ("member_casual",  lambda x: (x == "member").sum()),
            classic_count     = ("rideable_type",  lambda x: (x == "classic_bike").sum()),
            electric_count    = ("rideable_type",  lambda x: (x == "electric_bike").sum()),
            start_lat         = ("start_lat",      "median"),
            start_lng         = ("start_lng",      "median"),
        )
    )

    # Convert raw counts → shares (0–1 range)
    agg["casual_share"]        = agg["casual_count"]   / agg["daily_trip_count"]
    agg["member_share"]        = agg["member_count"]   / agg["daily_trip_count"]
    agg["classic_bike_share"]  = agg["classic_count"]  / agg["daily_trip_count"]
    agg["electric_bike_share"] = agg["electric_count"] / agg["daily_trip_count"]

    # Drop intermediate count columns — shares are sufficient
    agg.drop(
        columns=["casual_count", "member_count", "classic_count", "electric_count"],
        inplace=True,
    )
    return agg

print("aggregate_month() function defined.")

aggregate_month() function defined.


---
## Steps 4 & 5 — Main Loop: Load → Clean → Aggregate

The loop processes **one monthly zip at a time** to keep RAM usage low:
1. Open the zip and find the CSV inside
2. Load only the 9 needed columns via `usecols`
3. Call `clean_month()` → call `aggregate_month()`
4. **Delete** the raw and cleaned DataFrames before moving to the next month

The aggregated station-day panels (small) are collected for later combination.

In [8]:
monthly_panels: list[pd.DataFrame] = []

for month, zip_path in downloaded_zips.items():
    print(f"\n{'─'*55}")
    print(f"Processing {month}")
    print(f"{'─'*55}")

    # Open zip and locate the CSV inside
    with zipfile.ZipFile(zip_path, "r") as z:
        csv_files = [f for f in z.namelist() if f.endswith(".csv")]
        if not csv_files:
            print(f"  WARNING: no CSV found inside {zip_path} — skipping.")
            continue
        csv_name = csv_files[0]
        print(f"  Reading CSV: {csv_name}")

        with z.open(csv_name) as f:
            # usecols filters columns at read time → avoids loading unused data
            df_raw = pd.read_csv(
                f,
                usecols=lambda col: col in NEEDED_COLS,
                low_memory=False,
            )

    print(f"  Loaded   : {len(df_raw):,} rows × {df_raw.shape[1]} columns")

    # Step 4 — clean
    df_clean = clean_month(df_raw)
    print(f"  After cleaning : {len(df_clean):,} rows")
    del df_raw   # free raw data immediately

    # Step 5 — aggregate to station-day
    df_agg = aggregate_month(df_clean)
    print(f"  Station-day rows (aggregated) : {len(df_agg):,}")
    del df_clean   # free cleaned trip data

    monthly_panels.append(df_agg)

print(f"\nMonths processed: {len(monthly_panels)}")


───────────────────────────────────────────────────────
Processing 202601
───────────────────────────────────────────────────────
  Reading CSV: 202601-citibike-tripdata_2.csv
  Loaded   : 816,391 rows × 9 columns
    Rows dropped (missing critical fields) : 864
    Rows dropped (duplicate ride_id)       : 0
    Rows dropped (invalid duration <= 0)   : 0
  After cleaning : 815,527 rows
  Station-day rows (aggregated) : 45,343

───────────────────────────────────────────────────────
Processing 202602
───────────────────────────────────────────────────────
  Reading CSV: 202602-citibike-tripdata_1.csv
  Loaded   : 1,000,000 rows × 9 columns
    Rows dropped (missing critical fields) : 903
    Rows dropped (duplicate ride_id)       : 0
    Rows dropped (invalid duration <= 0)   : 0
  After cleaning : 999,097 rows
  Station-day rows (aggregated) : 51,811

Months processed: 2


---
## Step 6 — Station Filtering: Top 300 Busiest Stations

We first concatenate both months' aggregated panels, then count total trips per station **across both months combined**.  
Only the **top 300** stations by total trip volume are retained.

**Why 300?**  
This keeps the most informative stations, removes noise from very low-demand stations, and makes downstream modeling faster.

In [9]:
# Combine both months into one panel
combined = pd.concat(monthly_panels, ignore_index=True)
print(f"Combined station-day rows (before filter) : {len(combined):,}")
print(f"Unique stations (before filter)           : {combined['start_station_id'].nunique()}")

# Total trips per station across both months
station_totals = (
    combined
    .groupby("start_station_id")["daily_trip_count"]
    .sum()
    .sort_values(ascending=False)
)

top_station_ids = station_totals.head(TOP_N_STATIONS).index

print(f"\nTop-{TOP_N_STATIONS} station trip range:")
print(f"  Busiest station trips : {station_totals.iloc[0]:,}")
print(f"  #300 station trips    : {station_totals.iloc[TOP_N_STATIONS - 1]:,}")

# Apply filter
combined = combined[combined["start_station_id"].isin(top_station_ids)].copy()
print(f"\nStation-day rows (after filter) : {len(combined):,}")
print(f"Unique stations (after filter)  : {combined['start_station_id'].nunique()}")

Combined station-day rows (before filter) : 97,154
Unique stations (before filter)           : 2220

Top-300 station trip range:
  Busiest station trips : 8,122
  #300 station trips    : 1,946

Station-day rows (after filter) : 16,999
Unique stations (after filter)  : 300


---
## Step 7 — Merge Daily Weather Data

We join the weather table to every station-day row using `date` as the key (inner join).  
This means every row in the final dataset will have a complete set of weather values.  
We also report the overlap between trip dates and weather dates for transparency.

In [10]:
# Ensure both date columns are the same dtype (datetime64)
combined["date"]   = pd.to_datetime(combined["date"])
weather_df["date"] = pd.to_datetime(weather_df["date"])

trip_dates    = set(combined["date"].dt.date)
weather_dates = set(weather_df["date"].dt.date)
overlap       = trip_dates & weather_dates
only_trips    = trip_dates - weather_dates

print(f"Trip date range    : {min(trip_dates)} → {max(trip_dates)}")
print(f"Weather date range : {min(weather_dates)} → {max(weather_dates)}")
print(f"Overlapping dates  : {len(overlap)}")
if only_trips:
    print(f"WARNING — trip dates with no weather: {sorted(only_trips)}")

merged = combined.merge(weather_df, on="date", how="inner")
print(f"\nRows after merge : {len(merged):,}")
merged[["date", "mean_temperature", "precipitation_sum", "max_wind_speed"]].head()

Trip date range    : 2025-12-31 → 2026-02-28
Weather date range : 2026-01-01 → 2026-02-28
Overlapping dates  : 58
WARNING — trip dates with no weather: [datetime.date(2025, 12, 31)]

Rows after merge : 16,997


,date,mean_temperature,precipitation_sum,max_wind_speed
0,2026-01-09,5.4,3.6,16.8
1,2026-01-02,-4.4,0.3,14.4
2,2026-01-06,1.0,0.6,7.5
3,2026-01-12,1.3,0.0,22.6
4,2026-01-07,4.8,0.3,18.0


---
## Step 8 — Station Name Text Preprocessing

Station names are already lowercased from the cleaning step, so we can apply simple **whole-word regex matches** (`\b` word boundaries) to flag whether a station name contains a keyword.

Six binary indicator columns are created:

| Column | Keyword matched |
|--------|-----------------|
| `has_park`    | park |
| `has_pier`    | pier |
| `has_ferry`   | ferry |
| `has_plaza`   | plaza |
| `has_square`  | square |
| `has_station` | station |

These lightweight features capture basic station context (near a park, waterfront, transit hub, etc.) without heavy NLP.

In [11]:
KEYWORDS = {
    "has_park":    r"\bpark\b",
    "has_pier":    r"\bpier\b",
    "has_ferry":   r"\bferry\b",
    "has_plaza":   r"\bplaza\b",
    "has_square":  r"\bsquare\b",
    "has_station": r"\bstation\b",
}

print("Keyword indicator counts (station-day rows matched):")
for col, pattern in KEYWORDS.items():
    merged[col] = (
        merged["start_station_name"]
        .str.contains(pattern, regex=True, na=False)
        .astype(int)
    )
    n = merged[col].sum()
    unique_stations = merged.loc[merged[col] == 1, "start_station_id"].nunique()
    print(f"  {col:<16}: {n:>6,} rows  ({unique_stations} unique stations)")

Keyword indicator counts (station-day rows matched):
  has_park        :    985 rows  (17 unique stations)
  has_pier        :     58 rows  (1 unique stations)
  has_ferry       :      0 rows  (0 unique stations)
  has_plaza       :     59 rows  (1 unique stations)
  has_square      :     58 rows  (1 unique stations)
  has_station     :      0 rows  (0 unique stations)


---
## Step 9 — Build and Save Final Dataset

The 20 final columns are selected and renamed to match the project schema, then sorted by `station_id` and `date` and written to `output/citibike_station_day_clean.csv`.

**Final schema:**

| Column | Description |
|--------|-------------|
| `station_id` | Unique station identifier |
| `station_name` | Standardized station name |
| `date` | Calendar date |
| `start_lat` / `start_lng` | Median GPS coordinates |
| `daily_trip_count` | **Target variable** — trips starting from station on this day |
| `casual_share` / `member_share` | Rider type fractions |
| `classic_bike_share` / `electric_bike_share` | Bike type fractions |
| `mean_temperature` | Daily mean temp (°C) |
| `precipitation_sum` | Daily precipitation (mm) |
| `max_wind_speed` | Daily max wind speed (km/h) |
| `has_park` … `has_station` | Station name keyword indicators (0/1) |

In [12]:
FINAL_COLS = [
    "start_station_id",     # → renamed to station_id
    "start_station_name",   # → renamed to station_name
    "date",
    "start_lat",
    "start_lng",
    "daily_trip_count",
    "casual_share",
    "member_share",
    "classic_bike_share",
    "electric_bike_share",
    "mean_temperature",
    "precipitation_sum",
    "max_wind_speed",
    "has_park",
    "has_pier",
    "has_ferry",
    "has_plaza",
    "has_square",
    "has_station",
]

final_df = (
    merged[FINAL_COLS]
    .rename(columns={
        "start_station_id":   "station_id",
        "start_station_name": "station_name",
    })
    .sort_values(["station_id", "date"])
    .reset_index(drop=True)
)

out_path = OUT_DIR / "citibike_station_day_clean.csv"
final_df.to_csv(out_path, index=False)

print("="*60)
print("Pipeline complete!")
print("="*60)
print(f"  Output file  : {out_path}")
print(f"  Rows         : {len(final_df):,}")
print(f"  Columns      : {len(final_df.columns)}")
print(f"  Stations     : {final_df['station_id'].nunique()}")
print(f"  Date range   : {final_df['date'].min().date()} → {final_df['date'].max().date()}")
print(f"  Unique days  : {final_df['date'].nunique()}")
print()
print("Column list:", list(final_df.columns))

Pipeline complete!
  Output file  : output/citibike_station_day_clean.csv
  Rows         : 16,997
  Columns      : 19
  Stations     : 300
  Date range   : 2026-01-01 → 2026-02-28
  Unique days  : 58

Column list: ['station_id', 'station_name', 'date', 'start_lat', 'start_lng', 'daily_trip_count', 'casual_share', 'member_share', 'classic_bike_share', 'electric_bike_share', 'mean_temperature', 'precipitation_sum', 'max_wind_speed', 'has_park', 'has_pier', 'has_ferry', 'has_plaza', 'has_square', 'has_station']


---
## Preview — Final Dataset

In [13]:
print("First 5 rows:")
final_df.head()

First 5 rows:


,station_id,station_name,date,start_lat,start_lng,daily_trip_count,casual_share,member_share,classic_bike_share,electric_bike_share,mean_temperature,precipitation_sum,max_wind_speed,has_park,has_pier,has_ferry,has_plaza,has_square,has_station
0,3651.04,west drive & prospect park west,2026-01-02,40.661063,-73.979453,2,0.0,1.0,1.0,0.0,-4.4,0.3,14.4,1,0,0,0,0,0
1,3651.04,west drive & prospect park west,2026-01-03,40.661063,-73.979453,2,0.0,1.0,0.0,1.0,-3.5,0.0,10.3,1,0,0,0,0,0
2,3651.04,west drive & prospect park west,2026-01-04,40.661063,-73.979453,1,1.0,0.0,0.0,1.0,-2.2,0.3,12.1,1,0,0,0,0,0
3,3651.04,west drive & prospect park west,2026-01-05,40.661063,-73.979453,1,0.0,1.0,0.0,1.0,-1.8,0.8,12.6,1,0,0,0,0,0
4,3651.04,west drive & prospect park west,2026-01-06,40.661063,-73.979453,1,0.0,1.0,0.0,1.0,1.0,0.6,7.5,1,0,0,0,0,0


In [14]:
print("Column dtypes and null counts:")
summary = pd.DataFrame({
    "dtype":    final_df.dtypes,
    "non_null": final_df.notnull().sum(),
    "null":     final_df.isnull().sum(),
})
summary

Column dtypes and null counts:


,dtype,non_null,null
station_id,object,16997,0
station_name,object,16997,0
date,datetime64[ns],16997,0
start_lat,float64,16997,0
start_lng,float64,16997,0
daily_trip_count,int64,16997,0
casual_share,float64,16997,0
member_share,float64,16997,0
classic_bike_share,float64,16997,0
electric_bike_share,float64,16997,0


In [15]:
print("Descriptive statistics for numeric columns:")
final_df.describe().round(3)

Descriptive statistics for numeric columns:


,date,start_lat,start_lng,daily_trip_count,casual_share,member_share,classic_bike_share,electric_bike_share,mean_temperature,precipitation_sum,max_wind_speed,has_park,has_pier,has_ferry,has_plaza,has_square,has_station
count,16997,16997.000,16997.000,16997.000,16997.000,16997.000,16997.000,16997.000,16997.000,16997.000,16997.000,16997.000,16997.000,16997.0,16997.000,16997.000,16997.0
mean,2026-01-30 01:26:24.914984704,40.738,-73.985,53.018,0.076,0.924,0.285,0.715,-2.973,1.693,15.765,0.058,0.003,0.0,0.003,0.003,0.0
min,2026-01-01 00:00:00,40.661,-74.017,1.000,0.000,0.000,0.000,0.000,-14.300,0.000,4.400,0.000,0.000,0.0,0.000,0.000,0.0
25%,2026-01-16 00:00:00,40.723,-73.995,12.000,0.018,0.904,0.202,0.649,-7.500,0.000,12.100,0.000,0.000,0.0,0.000,0.000,0.0
50%,2026-01-30 00:00:00,40.738,-73.987,44.000,0.056,0.944,0.275,0.725,-1.800,0.000,15.300,0.000,0.000,0.0,0.000,0.000,0.0
75%,2026-02-13 00:00:00,40.755,-73.978,77.000,0.096,0.982,0.351,0.798,1.300,0.600,19.300,0.000,0.000,0.0,0.000,0.000,0.0
max,2026-02-28 00:00:00,40.808,-73.940,421.000,1.000,1.000,1.000,1.000,6.400,33.100,29.800,1.000,1.000,0.0,1.000,1.000,0.0
std,NaN,0.023,0.015,48.118,0.100,0.100,0.157,0.157,5.503,5.018,5.130,0.234,0.058,0.0,0.059,0.058,0.0
